# Lab 01 · Prompting and small evaluations

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## Give the model the policy, then test it

**So far:** in Lab 00 the model answered without seeing the policy, and it got the mixed cases wrong.

**New:** a *guided prompt* that contains the policy, a *parser* that rejects badly formatted replies, and a small *evaluation* that scores the model on five labelled cases.

A guided prompt has four labelled parts: the policy, one worked example, the case, and the exact reply format. JSON lets our code read the decision reliably. The one-sentence reason is for a person to check. Asking the model to check both criteria is a short, visible form of step-by-step reasoning. We do not ask for a long reasoning trace.

Step 5 is an optional live contrast: why a plain model can miss a letter count, how writing the steps down (chain-of-thought) usually fixes it, and how a *reasoning* model spends tokens reasoning before it answers. It needs `MODE` set to live and an `OPENROUTER_API_KEY`, and falls back to scripted answers otherwise.

**At a glance**

![Lab 01 at a glance: Labelled case (5 tests), then Guided prompt (policy + example), then Model (JSON reply), then Parse (parse_reply()), then Score (pass · fail · invalid), then ≥ 80%.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/01.svg)

## The idea in context

Early work with LLMs was mostly trial and error: change the wording, read one answer, change it again. Two habits turned this into engineering. First, prompts became structured: clear instructions, the relevant policy, a worked example (few-shot prompting), and an exact output format. Second, the output became data: a parser turns a reply into a value that code can check, and a small labelled test set turns "it looks right" into a score.

A model's answers vary, so one good answer proves little. A test set with edge cases, such as the boundary of exactly 8 people, shows where a prompt breaks, and a quality gate stops a change that makes things worse. Every later lab keeps this habit: never trust output that you have not checked.

| Key term | Meaning |
| --- | --- |
| **Few-shot prompting** | Including one or more worked examples in the prompt, so the model sees what a good answer looks like. |
| **Structured output** | A reply in a fixed format, such as JSON, that code can parse. |
| **Evaluation set** | Labelled cases with known answers, used to score a prompt or a system. |
| **Quality gate** | A check that stops the run when the score falls below a threshold. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Build a guided prompt

`guided_prompt(case)` fills the same four-part template for any case. The worked example deliberately uses a case where only **one** criterion holds (20 people, but no independent deploys), because that is exactly where Lab 00's model went wrong. `json.dumps(case)` writes the case as JSON, so the model sees `false` in the same style as the worked example.

In [ ]:
import json

POLICY = ("Choose microservices only if team_size >= 8 AND independent_deploys is true; "
          "otherwise choose modular monolith.")
WORKED_EXAMPLE = ('Case: {"team_size": 20, "independent_deploys": false}\n'
                  'Reply: {"decision": "modular monolith", "reason": "20 people, but no independent deploys."}')

def guided_prompt(case):
    return (f"Policy: {POLICY}\n\n"
            f"Worked example:\n{WORKED_EXAMPLE}\n\n"
            f"Case: {json.dumps(case)}\n"
            'Check both criteria, then reply with JSON only: {"decision": "...", "reason": "one sentence"}')

print(guided_prompt({"team_size": 3, "independent_deploys": False}))

## Step 2 · Parse the reply, and reject what breaks the format

A model can ignore the format. `parse_reply` returns the decision when the reply is valid JSON with a known label. Otherwise it raises a `ValueError` that says what is wrong. The `try`/`except` loop below tries it on one good reply and two bad ones. Rejecting bad output loudly is safer than guessing what the model meant.

In [ ]:
LABELS = ("modular monolith", "microservices")

def parse_reply(reply):
    """Return the decision from a JSON reply. Raise ValueError if the reply breaks the format."""
    try:
        data = json.loads(reply)
    except json.JSONDecodeError:
        raise ValueError("not JSON") from None
    decision = str(data.get("decision", "")).strip().lower()
    if decision not in LABELS:
        raise ValueError(f"unknown decision {decision!r}")
    return decision

for reply in ['{"decision": "Modular Monolith", "reason": "3 people."}', "Microservices!", '{"decision": "serverless"}']:
    try:
        print("accepted:", parse_reply(reply))
    except ValueError as error:
        print("rejected:", error)

## Step 3 · Write a labelled test set

Each test has a case, the label the policy expects, and a note on why the case is in the set. Together the five cases cover both criteria, each criterion on its own, and the boundary value of exactly 8 people. In practice mode we also need a scripted reply for each test; one scripted label is deliberately wrong, so you can see what a failing row looks like.

In [ ]:
tests = [  # (case, expected label, why this case is in the set)
    ({"team_size": 3,  "independent_deploys": False}, "modular monolith", "neither criterion"),
    ({"team_size": 12, "independent_deploys": True},  "microservices",    "both criteria"),
    ({"team_size": 9,  "independent_deploys": False}, "modular monolith", "big team only"),
    ({"team_size": 5,  "independent_deploys": True},  "modular monolith", "independent deploys only"),
    ({"team_size": 8,  "independent_deploys": True},  "microservices",    "boundary: exactly 8"),
]
# Scripted practice labels; the third one is a deliberate mistake.
scripted = ["modular monolith", "microservices", "microservices", "modular monolith", "microservices"]

## Step 4 · Score the prompt

`zip(tests, scripted)` pairs each test with its scripted reply. Every reply ends up in one of three buckets: **pass** (right label), **fail** (wrong label), or **invalid** (the parser rejected it). The final `assert` is a *quality gate*: the run stops with an error if fewer than 80% of the tests pass.

In [ ]:
counts = {"pass": 0, "fail": 0, "invalid": 0}
for (case, expected, why), label in zip(tests, scripted):
    example = json.dumps({"decision": label, "reason": "Scripted practice reply."})
    try:
        decision = parse_reply(ask(guided_prompt(case), example=example))
        verdict = "pass" if decision == expected else "fail"
    except ValueError as error:
        decision, verdict = f"({error})", "invalid"
    counts[verdict] += 1
    print(f"{verdict.upper():<8} {why:<25} got {decision!r}, expected {expected!r}")
accuracy = counts["pass"] / len(tests)
print(counts, f"accuracy: {accuracy:.0%}")
assert accuracy >= 0.8, "Below the 80% quality gate. Read the FAIL and INVALID rows."

## Step 5 (live) · Tokens, chain-of-thought, and a reasoning model

Here is a famous failure: ask a non-reasoning model how many of the letter _R_ are in "strawberry". A model reads text as **tokens** — small chunks, and a short familiar word can be a single token — so it recognises the word but does not reliably *count* the letters inside it.

This cell runs three calls on OpenRouter. The first is a plain **old** model on its own. The second adds a **chain-of-thought** prompt that names every letter, turning the hidden guess into an explicit count. The third keeps the simple prompt but uses a **reasoning** model, which writes its own letter-by-letter count before it answers — and that inner writing is charged as output tokens.

It needs an `OPENROUTER_API_KEY` in Colab Secrets (never paste the key into a cell). Without one it prints scripted answers.

In [ ]:
import os, json, urllib.request
API = "https://openrouter.ai/api/v1/chat/completions"
SIMPLE = 'How many R\'s are in "strawberry"? Answer with a number.'
COT = 'How many R\'s are in "strawberry"? First say the letters S-T-R-A-W-B-E-R-R-Y one by one, then answer.'
OLD, REASON = "openai/gpt-3.5-turbo-0613", "qwen/qwen3-235b-a22b-thinking-2507"

def chat(model, prompt):
    body = json.dumps({"model": model, "messages": [{"role": "user", "content": prompt}], "max_tokens": 900}).encode()
    req = urllib.request.Request(API, body, {"Authorization": "Bearer " + os.environ["OPENROUTER_API_KEY"], "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.load(r)["choices"][0]["message"]

if os.getenv("OPENROUTER_API_KEY") and os.environ.get("MODE") == "live":
    plain, cot, rsn = chat(OLD, SIMPLE), chat(OLD, COT), chat(REASON, SIMPLE)
    print("old, simple:   ", plain.get("content"))
    print("old, CoT:      ", cot.get("content"))
    print("reason, simple:", rsn.get("content"))
    print("reason trace:  ", (rsn.get("reasoning") or "")[-70:])
else:
    print("Practice or no key: scripted answers (2 / 3 / 3). Live needs MODE=live and OPENROUTER_API_KEY.")

## What you should see

Step 2 prints `accepted: modular monolith`, then two `rejected:` lines. Step 4 prints five rows; four are `PASS`, and the `big team only` row is `FAIL` because its scripted reply says `microservices`, for accuracy `80%` on the gate.

Step 5 compares three runs on the same question, "how many R's are in strawberry?":

| Setup | Typical answer |
| --- | --- |
| **old** model, simple prompt | `2` — it misses the two R's in the middle |
| **old** model, chain-of-thought prompt | `3` — naming the letters turns a guess into a count |
| **reasoning** model, simple prompt | `3` — it re-counts silently before answering |

**Why a small model gets it wrong.** The prompt is split into **tokens**, and a short familiar word like "strawberry" can be a single token. The model recognises the word, but nothing in that token makes each letter separately available, so counting letters is the one thing it tends to guess. This is a property of how the model reads text, not a sign the model is broken.

**Why chain-of-thought usually fixes it.** Spelling the word out forces the model to emit `r` at positions 3, 8 and 9 one at a time. Each letter becomes a separate token the model wrote, in order; the count is now a record of what it already emitted. The model does more work, so it tends to answer correctly.

**Why a reasoning model spends tokens before it answers.** The reasoning model re-writes that same count for itself, inside a `reasoning` field, and only then prints the final `3` in the `content` field. That inner reasoning is charged as **output tokens**, so the same-looking short answer costs far more `completion_tokens` — and if your `max_tokens` budget is too small, the model runs out *mid-reasoning* and never emits the final answer at all. A reasoning model's advantages are not free: they buy a checkable trace at the price of tokens and latency.

Practice scores only test evaluation code, not a model; the Step 5 numbers are scripted without a key. In live mode the score shows how this prompt did on these five cases, and Step 5 shows real model answers. Five cases cannot prove a prompt is good in general.

## Try it

1. Fix the deliberate mistake: change the third scripted label to `"modular monolith"`. Accuracy becomes 100%.
2. Change **two** scripted labels to wrong labels and confirm the gate stops the run below 80%.
3. In Step 5, swap `strawberry` for `raspberry` (3 r's) or `Mississippi` (4 i's). Which model still gets it right?
4. Cut the reasoning model's `max_tokens` to 60 and re-run: what does the `reasoning` field show, and why is `content` empty?
5. In live mode, compare this score with Lab 00's agreement count. Which part of the guided prompt made the difference?

## Takeaway

A clear prompt, a strict parser, and a labelled test set turn "the answer looks right" into a number you can track. But some failures come from *how* the model reads tokens, not from a bad prompt: a word its tokeniser treats as one piece hides its letters. Naming the steps you want (chain-of-thought) or choosing a reasoning model fixes that — at the cost of more tokens and latency, and a reasoning model's hidden trace consumes your output budget. Every later lab keeps this habit: it checks the model's output before trusting it.

## Read further

- [Chain-of-thought prompting](https://arxiv.org/abs/2201.11903)
- [Prompt engineering](https://developers.openai.com/api/docs/guides/prompt-engineering)
- [OpenRouter model settings](https://openrouter.ai/docs/features/setting-parameters)